<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Ek E: Yığınlama ve işlem hacmi odaklı çalıştırma

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- Ana bölümler boyunca genellikle her seferinde tek bir örneği işliyoruz
- Bu, kodu derli toplu ve anlaşılması daha kolay tutuyor
- Ayrıca kodun çalıştırılması zaten çok maliyetli; donanım ve kaynak kısıtları nedeniyle yığınlama desteği eklemek pek fayda sağlamazdı
- Ancak bazı bağlamlarda kodu yığınlı kipte çalıştırabilmek yine de faydalıdır
- Bu ek, yığınlı çalıştırmanın ardındaki genel fikri açıklar ve ek materyallerdeki kodu kullanarak farklı bölümlerde bunun nasıl kullanılacağını gösterir

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F01_raschka.webp" width="400px">

&nbsp;
## E.1 Yığınlama neden yardımcı olur

- İki farklı başarım hedefi vardır:
  - gecikme: tek bir istem için yanıtı ne kadar hızlı aldığımız;
  - işlem hacmi: belirli bir sürede kaç istem işleyebildiğimiz.
- Tek örnekli üretim, çoğu zaman gecikmeyi en aza indirmek ve kod hata ayıklamak için en iyisidir
- Yığınlama ise öncelikle işlem hacmini hedefler
- MATH-500 üzerinde yüzlerce problemi değerlendirmek, çok sayıda öz tutarlılık örneği üretmek ya da çok sayıda denetimli örnek üzerinde eğitim yapmak istiyorsak, uygun donanımda yığınlama toplam çalışma süresini belirgin biçimde azaltabilir
  - Yine de yığınlamanın her cihazda daha hızlı olacağı garanti değildir
  - CPU'lardaki küçük modeller ya da daha az optimize edilmiş bazı GPU'lar yığınlamadan yararlanmayabilir; hatta yavaşlama bile görebiliriz, çünkü ek dolgu ve yığınlama yükü paralellikten gelen kazancı dengeleyebilir

&nbsp;
## E.2 Yığınlı üretimi çalıştırmak

- Yığınlamadaki başlıca teknik engel, istemlerin genellikle farklı uzunluklarda olmasıdır
- Örneğin bir matematik problemi 40 token'a ayrılırken bir diğeri 120 token'a ayrılabilir
- PyTorch'taki tensörlerin dikdörtgen şekillerde olması gerektiğinden, daha kısa dizileri tek bir yığın tensörüne sığacak şekilde dolduruyoruz

- Kavramsal olarak bu, yığınlı üretimi tek istemli üretime göre uygulaması çok daha zor kılar
- Ana bölümde `reasoning_from_scratch.qwen3` içindeki `Qwen3Model` sınıfını kullandık (Ek C'de anlatılan Qwen3 uygulamasını kullanır)
- Yığınlı üretim için, dolgu token'larını vb. takip etmemiz gerektiğinden `reasoning_from_scratch.qwen3_batched` içinde ayrı bir `Qwen3Model` sınıfı bulunur (kaynak kod ek materyallerde https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3_batched.py adresinde görüntülenebilir)

- Yığınlı üretim araçlarının kullanımını göstermek için somut bir örneğe bakalım
- Ana bölümlerde kullandığımıza benzer, tek dizili bir metin üretme örneğiyle başlıyoruz
- Burada bunu iki isteme (`["2+2?", "3+3=6?"]`) ardışık olarak uyguluyoruz:

In [2]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- Aşağıda, `reasoning_from_scratch.qwen3_batched` içinden yığınlamayı destekleyen benzer bir kod kullanacağız
- Ancak yığınlı sürümün akış (streaming) desteklemediğini unutmayın; yani sonuçların kodu çözülüp yazdırılmadan önce hepsinin üretilmesini beklememiz gerekir
- Burada yığınlı üretim sol dolgu kullanır; bu, bir sonraki bölümde açıklanacak
- Şimdilik (içeride nasıl çalıştığına geçmeden önce) nasıl kullanıldığını göstermek için bir kullanım örneğiyle başlayalım

In [3]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


- Görüldüğü gibi sonuçlar öncekiyle tamamen aynı
- Fark, bu sonuçların `generate_text_basic_batched_cache` aracılığıyla paralel olarak üretilmiş olmasıdır
- Bir sonraki bölüm bunun arka planda nasıl çalıştığını kısaca açıklıyor

- Daha da optimize edilmiş bir kod uygulaması, `generate_text_basic_batched_cache` yerine `generate_text_basic_batched_cache_stop` kullanır
- `generate_text_basic_batched_cache`, her kod çözme adımında etkin yığındaki her satırı tutar 
- `generate_text_basic_batched_cache_stop` ise biten satırları etkin hesaplama yığınından çıkarır (içeride uygulaması daha karmaşıktır, ancak başarımı optimize edebilir
- Bu, aşağıdaki şekilde gösterilmiştir

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F02_raschka.webp?1" width="500px">

- Yan not: Qwen3'te `<eos>` token'ları `<|endoftext|>` şeklindedir; ancak şekil görsel derli topluluk için `<eos>` kullanıyor

&nbsp;
## E.3 Dolgu ve dikkat maskeleri

- Tek örnekli kipte, `"2+2?"` gibi kısa bir istemi token'lara ayırırsak, bunu modele `(1, 4)` şeklinde basit bir tensör olarak verebiliriz:
  - `input_ids = torch.tensor([[17, 10, 17, 30]])`

- Model içeride standart bir nedensel dikkat maskesi oluşturur; böylece her konum yalnızca kendisine ve önceki token'lara dikkat edebilir
- Öz-dikkate yabancıysanız, daha fazla arka plan bilgisi sunan bir yazım var: https://magazine.sebastianraschka.com/p/understanding-and-coding-self-attention
- Kavramsal olarak bu maske şöyle görünür:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F03_raschka.webp" width="400px">

- `1` "maskelenmiş", `0` ise "izinli" anlamına gelir
- Yani ilk token sonraki konumlara ileri bakamaz, ikinci token yalnızca ilk iki konuma bakabilir vb.
- Bu, standart özbağlanımlı maskeleme örüntüsüdür

- Yığınlama durumu değiştirir; çünkü farklı istemler genellikle farklı uzunluklardadır
- `"2+2?"` istemini biraz daha uzun olan `"3+3=6?"` istemiyle birlikte işlediğimizi varsayalım
- PyTorch tensörleri dikdörtgen olmak zorunda olduğundan, daha kısa satırın daha uzun olanla eşleşecek şekilde doldurulması gerekir
- Burada bu, sol dolgu ile yapılır:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-e/Appendix_E_F04_raschka.webp" width="500px">

- İçeride ek bir `attn_mask` tuttuğumuzu unutmayın; bu yalnızca doldurulmuş konumları takip etmek içindir
- Bu `attn_mask` içinde `True` doldurulmuş, `False` doldurulmamış anlamına gelir
- Bu ek `attn_mask` maskesini, nedensel maskede dolgu token kimliklerine karşılık gelen token'ları belirlemek için kullanıyoruz
- Doldurulmuş anahtarları maskelemek ve doldurulmuş sorguları sıfırlamak, yığınlamanın tek örnekli çalıştırmaya benzer davranmasını sağlayan önemli adımlardır

- Bu arada `<|endoftext|>` token'ını kullanıyoruz; ancak bu aslında önemli değil, çünkü karşılık gelen token konumları zaten yok sayılıyor

In [4]:
print(tokenizer.pad_token_id)

151643


In [5]:
print(tokenizer.decode([151643]))

<|endoftext|>


&nbsp;
## E.4 Bölüm 3: yığınlı MATH-500 değerlendirmesi

- Ek materyaller, 3. bölümde uygulanan değerlendirme yöntemi için, 6. bölümde yaptığımıza benzer şekilde indirip kullanabileceğimiz bir betik içerir:

In [7]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)

evaluate_math500.py: 3.5 KB
math500_test.json: 462.1 KB


- Ardından çalıştırmak için bir kod terminalinde şu komutu yürütebiliriz (uv kullanıcısı değilseniz `uv run` yerine `python` yazın):

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --which_model "reasoning"
```

- Bonus materyal, daha önce tartıştığımız yığınlama yöntemini uygulayan yığınlı üretim sürümünü de içerir
- İndirme öncekine benzer; yalnızca `evaluate_math500.py` yerine `evaluate_math500_batched.py` yazıyoruz

In [8]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

evaluate_math500_batched.py: 8.3 KB


- Kullanımı da yığınsız sürüme benzer; yalnızca artık LLM'in kaç istem ve yanıtı paralel işleyeceğini belirtmek için ek bir `--batch_size` argümanı veriyoruz

```bash
uv run evaluate_math500_batched.py \
  --dataset_size 500 \
  --which_model "reasoning" \
  --batch_size 64
```

- İdeal yığın boyutu donanımınızın neyi kaldırabildiğine bağlıdır; 64 yığın boyutu yaklaşık 23,39 GB RAM kullanır (yığınsız betik yaklaşık 1,84 GB RAM kullanır)
- Başarım farkını ekin sonuna doğru karşılaştırıp tartışacağız

&nbsp;
## E.5 Bölüm 4: yığınlı öz tutarlılık örneklemesi

- 4. bölümdeki öz tutarlılık örneklemesini uygulayan isteğe bağlı `self_consistency_math500_batched.py` betiği, farklı istemleri tek bir doldurulmuş tensörde karıştırmaz
-  Bunun yerine aynı istemi `num_samples` kez tekrarlar ve öz tutarlılık oylaması için birkaç devamı paralel olarak örnekler
- Her satır aynı istem uzunluğundan başladığı için, eşit istem uzunluklarında dolguya gerek olmadığından bu betik reasoning_from_scratch.qwen3_batched yerine reasoning_from_scratch.qwen3 içindeki normal `Qwen3Model` sınıfını kullanır

- Betiği şöyle indirebiliriz:

In [ ]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/self_consistency_math500_batched.py"
)

- Yığınsız sürümü indirmek için yukarıdaki dosya adındaki `"_batched"` ifadesini çıkarmanız yeterli
- Betiği şöyle çalıştırabiliriz (yığınsız betiğin söz dizimi aynıdır)

```bash
uv run self_consistency_math500_batched.py \
  --which_model base \
  --temperature 0.9 \
  --top_p 0.9 \
  --num_samples 3 \
  --dataset_size 500 \
  --prompt_suffix "\n\nExplain step by step."
```

- Başarımla ilgili daha fazlası bu ekin sonunda

&nbsp;
## E.6 Bölüm 6: yığınlı GRPO rollout'ları

- 5. bölümdeki öz iyileştirme, kendisi yığınlamadan yararlanmayan ardışık bir tekniktir
- Birden çok girdi için öz iyileştirme döngüleri paralel çalıştırılabilirdi; ancak bunu uygulamak önemsiz değildir ve bu nedenle ek materyalin parçası değildir
- Bunun yerine 6. bölümdeki RLVR'nin yığınlı bir sürümüyle devam ediyoruz
- 6. bölümde farklı rollout'lar için aynı istemi kullanıyoruz; dolayısıyla burada dolguya gerek yok; bu nedenle E.5 kısmına benzer şekilde kod, `reasoning_from_scratch.qwen3` içindeki normal `Qwen3Model` sınıfını kullanır
- İlgili betikler şöyle alınabilir:

In [ ]:
# Yığınsız sürüm
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

# Yığınlı sürüm
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched.py"
)

# GPU destekli yığınlı sürüm
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl_batched_fsdp.py"
)

```bash
uv run rlvr_grpo_original_no_kl_batched.py \
  --num_rollouts 8 \
  --steps 100 \
  --batch_size 4 \
  --max_new_tokens 1024
```

- Mevcut betikte `--batch_size`, bir adım içinde kaç rollout'un paralel üretileceğini denetler
- Bu, işlem hacmini artırır; ancak bellek baskısını da artırır, bu yüzden pratikte `--num_rollouts` ya da `--max_new_tokens` değerlerini düşürmeniz gerekebilir
- Birden çok GPU'nuz varsa, FSDP çeşidi aynı örüntüyü izler ve `--num_gpus` ekler
- Başarım tartışmasına yine bu ekin sonunda döneceğiz
- Bu yazının yazıldığı tarihte 7. bölüm betiklerinin yığınlı sürümleri henüz ek materyallerde yok, ancak zamanla eklenecek; kavramsal olarak 6. bölüm betiklerine benzer çalışacaklar

&nbsp;
## E.7 Bölüm 8: yığınlı damıtma

- 8. bölüm, damıtma örneklerinin farklı istem ve yanıt uzunluklarına sahip olması nedeniyle 3. bölümdeki dolgu duyarlı biçime geri döner
- Betiği ve örnek eğitim veri kümesini şöyle indirebilirsiniz:

In [9]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

distill_batched.py: 17.9 KB
deepseek-r1-math-train.json: 107538.0 KB


- Yığınsız sürüm için dosya adındaki `"_batched"` ifadesini çıkarın
- Betiği şöyle çalıştırabiliriz:

```bash
uv run distill_batched.py \
  --data_path deepseek-r1-math-train.json \
  --dataset_size 12000 \
  --validation_size 10 \
  --epochs 2 \
  --use_think_tokens \
  --max_seq_len 1024 \
  --batch_size 4
```

&nbsp;
## E.8 Tek dizili üretime karşı yığınlı üretim

- Aşağıdaki tablo, yukarıdaki betikler için çalışma süresi ve RAM kullanım değerlerini özetler

| Satır | Betik                                  | Yığın boyutu | RAM      | H100 toplam süre (dk) | DGX Spark toplam süre (dk) |
|-----|------------------------------------------|------------|----------|------------------------|-----------------------------|
| 1   | evaluate_math500.py                      | -          | 1.8 GB   | 90.0                   | 174.7                       |
| 2   | evaluate_math500_batched.py              | 64         | 23.39 GB | 16.0                   | 108.4                       |
|     |                                          |            |          |                        |                             |
| 3   | self_consistency_math500.py              | -          | 1.79 GB  | 252.0                  | 340.8                       |
| 4   | self_consistency_math500_batched.py      | 3          | 2.45 GB  | 129.0                  | 243.3                       |
|     |                                          |            |          |                        |                             |
| 5   | rlvr_grpo_original_no_kl.py              | -          | 43.35 GB | 68.0                   | 63.7                        |
| 6   | rlvr_grpo_original_no_kl_batched.py      | 4          | 44.91 GB | 19.0                   | 23.1                        |
|     |                                          |            |          |                        |                             |
| 7   | distill.py                              | -          | 8.29 GB  | 10.9                   | 32.8                        |
| 8   | distill_batched.py                      | 4          | 8.34 GB  | 9.1                    | 28.2                        |